# Raw dataset exploration — training and test

**Purpose.** Describe the two raw CSV files well enough to make modelling decisions: class
balance, missing data versus survey response codes, cardinality, raw value formats,
association with the target, and train/test comparability.

**No transformations.** Nothing here is filtered, recoded, imputed, or written back to
disk. Every table and plot is computed in memory from the raw values. In particular,
`"Sem resposta"` and `"Não aplicável"` are survey answers and are kept as ordinary
categories throughout.

**Sources.** `data/dataset_train.csv` (includes `AGE_RANGE`) and
`data/dataset_test_X.csv` (target withheld).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import chi2_contingency

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 180)
pd.set_option("display.max_rows", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

RNG = 42
TARGET = "AGE_RANGE"
SENTINELS = ["Sem resposta", "Não aplicável"]

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

train = pd.read_csv(PROJECT_ROOT / "data" / "dataset_train.csv", low_memory=False)
test = pd.read_csv(PROJECT_ROOT / "data" / "dataset_test_X.csv", low_memory=False)

print(f"train: {train.shape[0]:,} rows x {train.shape[1]} columns")
print(f"test:  {test.shape[0]:,} rows x {test.shape[1]} columns")
print(f"memory: train {train.memory_usage(deep=True).sum() / 1024**2:,.1f} MB | "
      f"test {test.memory_usage(deep=True).sum() / 1024**2:,.1f} MB")

## 1. Structural integrity

Before looking at any feature, check the things that silently break models: mismatched
columns, duplicated rows, identifiers that are not actually identifiers, and columns that
carry no information.

In [ ]:
def column_kind(series):
    # Classify a column as numeric or string for reporting purposes only.
    if pd.api.types.is_numeric_dtype(series):
        return "numeric"
    if pd.api.types.is_string_dtype(series):
        return "string"
    return str(series.dtype)


kind_counts = pd.Series(
    [column_kind(train[column]) for column in train.columns], name="columns"
).value_counts()
display(kind_counts.to_frame())

train_only = [column for column in train.columns if column not in test.columns]
test_only = [column for column in test.columns if column not in train.columns]
print("columns only in train:", train_only)
print("columns only in test:", test_only or "none")
print("duplicate rows (train):", int(train.duplicated().sum()))
print("duplicate rows (test): ", int(test.duplicated().sum()))
print("PID unique (train/test):", train["PID"].is_unique, test["PID"].is_unique)
print("HHID unique (train/test):", train["HHID"].is_unique, test["HHID"].is_unique)
print("PID overlap:", len(set(train["PID"]) & set(test["PID"])))
print("HHID overlap:", len(set(train["HHID"]) & set(test["HHID"])))
print("households by number of rows (train):")
display(train.groupby("HHID").size().value_counts().rename("households").to_frame().T)

In [ ]:
n_levels = train.nunique(dropna=False)
top_value = pd.Series(
    [train[column].value_counts(dropna=False).index[0] for column in train.columns],
    index=train.columns,
)
top_share = pd.Series(
    [
        train[column].value_counts(dropna=False).iloc[0] / len(train)
        for column in train.columns
    ],
    index=train.columns,
)
column_profile = pd.DataFrame(
    {"n_levels": n_levels, "top_value": top_value, "top_share": top_share}
)

print("constant columns:")
display(column_profile[column_profile["n_levels"] <= 1])
print("near-constant columns (one value covers more than 99% of rows):")
display(
    column_profile[column_profile["top_share"] > 0.99]
    .sort_values("top_share", ascending=False)
)

## 2. Target distribution

`AGE_RANGE` is a 16-class target and the competition metric is macro F1, so every class
counts equally. The relevant number is not overall accuracy but how few examples the
smallest classes have.

In [ ]:
target_counts = train[TARGET].value_counts().sort_index()
target_table = pd.DataFrame(
    {
        "count": target_counts,
        "percent": (100 * target_counts / len(train)).round(2),
        "relative_to_smallest": (target_counts / target_counts.min()).round(2),
    }
)
display(target_table)

print(f"classes: {len(target_counts)}")
print(f"smallest class: {target_counts.idxmin()} with {target_counts.min():,} rows")
print(f"largest class:  {target_counts.idxmax()} with {target_counts.max():,} rows")
print(f"imbalance ratio: {target_counts.max() / target_counts.min():.2f}:1")
print(f"majority-class accuracy baseline: {100 * target_counts.max() / len(train):.2f}%")
print(f"random macro-F1 reference: {1 / len(target_counts):.4f}")

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(target_counts.index.astype(str), target_counts.values, color="#4477aa")
ax.set_title("AGE_RANGE class counts")
ax.set_xlabel("AGE_RANGE code")
ax.set_ylabel("rows")
plt.tight_layout()
plt.show()

## 3. Real missing values versus survey codes

`NaN` means the cell is absent. `"Sem resposta"` and `"Não aplicável"` are answers:
non-response and routing (a question that does not apply to that person). Treating them
as `NaN` would erase information, especially for skip patterns that are themselves
age-related.

In [ ]:
missing = train.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
print(f"columns containing at least one NaN: {len(missing)}")
display(pd.DataFrame({"missing": missing, "percent": (100 * missing / len(train)).round(2)}))

In [ ]:
sentinel_counts = pd.DataFrame(
    {token: (train == token).sum() for token in SENTINELS}
)
sentinel_counts["any_sentinel"] = sentinel_counts.sum(axis=1)
sentinel_counts["any_sentinel_pct"] = (
    100 * sentinel_counts["any_sentinel"] / len(train)
).round(2)

print("columns containing each token:")
display(sentinel_counts.gt(0).sum().rename("columns").to_frame().T)
print("top columns by sentinel volume:")
display(sentinel_counts.sort_values("any_sentinel", ascending=False).head(20))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = {"Sem resposta": "#cc6677", "Não aplicável": "#88ccee"}
for axis, token in zip(axes, SENTINELS):
    top = sentinel_counts[token].sort_values(ascending=False).head(15).iloc[::-1]
    axis.barh(top.index, top.values, color=colors[token])
    axis.set_title(f"Rows answering {token!r}")
    axis.set_xlabel("rows")
plt.tight_layout()
plt.show()

## 4. Cardinality and identifier audit

Cardinality drives encoding choices. A categorical with a handful of levels can be
one-hot encoded; a column with one hundred levels needs a different plan. Columns whose
cardinality equals the row count are identifiers, not features.

In [ ]:
cardinality = column_profile.copy()
cardinality.insert(0, "kind", [column_kind(train[column]) for column in train.columns])
cardinality["n_missing"] = train.isna().sum()

print("level-count summary:")
display(cardinality["n_levels"].describe().to_frame().T)
print("highest-cardinality columns:")
display(cardinality.sort_values("n_levels", ascending=False).head(20))
print("columns with two levels or fewer (excluding the target):")
display(
    cardinality[
        (cardinality["n_levels"] <= 2) & (cardinality.index != TARGET)
    ].sort_values("n_levels")
)

In [ ]:
numeric_columns = [
    column for column in train.columns if pd.api.types.is_numeric_dtype(train[column])
]
display(train[numeric_columns].describe().T)
print("REFYEAR values:", train["REFYEAR"].value_counts().to_dict())
print("REFMONTH range:", int(train["REFMONTH"].min()), "to", int(train["REFMONTH"].max()))

## 5. Raw value formats

Some columns look numeric but are strings. Casting them blindly can destroy leading
zeros and fixed-width codes, and some columns mix numbers with range labels. This section
lists those columns; it does not change them.

In [ ]:
def non_sentinel_values(series):
    # Return non-null values with survey response codes removed, as strings.
    values = series.dropna().astype(str)
    return values[~values.isin(SENTINELS)]


numeric_like_rows = []
for column in train.columns:
    if not pd.api.types.is_string_dtype(train[column]):
        continue
    values = non_sentinel_values(train[column])
    if values.empty:
        continue
    if values.str.fullmatch(r"-?\d+(?:\.\d+)?").all():
        numeric_like_rows.append(
            {
                "column": column,
                "n_values": len(values),
                "n_unique": values.nunique(),
                "leading_zero_values": int(values.str.fullmatch(r"0\d+").sum()),
                "example_values": ", ".join(values.drop_duplicates().head(3)),
            }
        )

numeric_like = pd.DataFrame(numeric_like_rows).sort_values(
    "n_unique", ascending=False
)
print(f"string columns whose non-sentinel values are all numeric: {len(numeric_like)}")
display(numeric_like)

In [ ]:
mixed_format_rows = []
for column in train.columns:
    if not pd.api.types.is_string_dtype(train[column]):
        continue
    values = non_sentinel_values(train[column])
    if values.empty:
        continue
    numeric_share = values.str.fullmatch(r"-?\d+(?:\.\d+)?").mean()
    if 0 < numeric_share < 1:
        mixed_format_rows.append(
            {
                "column": column,
                "numeric_share": round(float(numeric_share), 4),
                "examples": " | ".join(values.drop_duplicates().head(4)),
            }
        )

print("string columns mixing numeric and non-numeric values:")
display(pd.DataFrame(mixed_format_rows))

## 6. Association with `AGE_RANGE`

Cramér's V measures how strongly each low-cardinality categorical is associated with the
target, with a chi-square test for reference. It is descriptive: it ignores interactions
and it is not the same as model importance. High-cardinality columns are excluded rather
than silently binned, because binning would be a transformation.

In [ ]:
def cramers_v(feature, target):
    # Return Cramér's V and the chi-square p-value for two categorical series.
    table = pd.crosstab(feature, target, dropna=False)
    if table.shape[0] < 2 or table.shape[1] < 2:
        return np.nan, np.nan
    chi2, p_value, _, _ = chi2_contingency(table, correction=False)
    n_observations = table.values.sum()
    v = np.sqrt((chi2 / n_observations) / (min(table.shape) - 1))
    return float(v), float(p_value)


association_rows = []
for column in train.columns:
    if column in (TARGET, "PID", "HHID"):
        continue
    n_levels_column = train[column].nunique(dropna=False)
    if n_levels_column < 2 or n_levels_column > 60:
        continue
    v, p_value = cramers_v(train[column], train[TARGET])
    association_rows.append(
        {
            "column": column,
            "n_levels": n_levels_column,
            "cramers_v": v,
            "p_value": p_value,
        }
    )

association = pd.DataFrame(association_rows).sort_values(
    "cramers_v", ascending=False
)
print(f"features evaluated: {len(association)}")
display(association.head(30))

top_association = association.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top_association["column"], top_association["cramers_v"], color="#4477aa")
ax.set_xlabel("Cramér's V with AGE_RANGE")
ax.set_title("Top 20 raw categorical associations")
plt.tight_layout()
plt.show()

## 7. Target profile of key socioeconomic and health fields

These row-normalised tables show the age-class mix inside each category. They are useful
for spotting routing effects (for example, questions only asked above a certain age) and
for deciding which variables deserve ordinal treatment.

In [ ]:
CROSSTAB_COLUMNS = [
    "SEX",
    "HS1",
    "HS3",
    "MAINSTAT",
    "HATLEVEL",
    "REGION",
    "HHINCOME",
]

for column in CROSSTAB_COLUMNS:
    table = pd.crosstab(train[column], train[TARGET], normalize="index") * 100
    table = table.round(1)
    table["n"] = train[column].value_counts()
    print(f"\n{column} (row percentages, target columns 0-15)")
    display(table)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
for axis, column in zip(axes, ["SEX", "HS1"]):
    shares = pd.crosstab(train[column], train[TARGET], normalize="index")
    bottom = np.zeros(len(shares))
    for target_code in shares.columns:
        axis.bar(
            shares.index.astype(str),
            shares[target_code].values,
            bottom=bottom,
            label=str(target_code),
        )
        bottom += shares[target_code].values
    axis.set_title(f"{column} stacked by AGE_RANGE")
    axis.set_ylabel("share of rows within category")
axes[1].legend(
    title="AGE_RANGE", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=7, ncol=1
)
plt.tight_layout()
plt.show()

## 8. Train/test comparability

The test set is scored on Macro F1, so unseen categories and distribution shifts matter.
Total variation distance (TVD) is a simple summary: 0 means identical distributions, 1
means disjoint support. Levels present only in test are listed separately, because a
model that has never seen them cannot assign them a learned coefficient.

In [ ]:
drift_rows = []
for column in test.columns:
    if train[column].nunique(dropna=False) > 60:
        continue
    train_share = train[column].value_counts(normalize=True, dropna=False)
    test_share = test[column].value_counts(normalize=True, dropna=False)
    levels = train_share.index.union(test_share.index)
    tvd = 0.5 * np.abs(
        train_share.reindex(levels, fill_value=0)
        - test_share.reindex(levels, fill_value=0)
    ).sum()
    drift_rows.append({"column": column, "levels": len(levels), "tvd": float(tvd)})

drift = pd.DataFrame(drift_rows).sort_values("tvd", ascending=False)
print("largest train/test distribution distances:")
display(drift.head(20))

top_drift = drift.head(20).iloc[::-1]
fig, ax = plt.subplots(figsize=(8, 7))
ax.barh(top_drift["column"], top_drift["tvd"], color="#cc6677")
ax.set_xlabel("total variation distance (train vs test)")
ax.set_title("Top 20 categorical distribution shifts")
plt.tight_layout()
plt.show()

In [ ]:
unseen_rows = []
for column in test.columns:
    if column in ("PID", "HHID"):
        continue
    train_levels = set(train[column].dropna().unique())
    test_levels = set(test[column].dropna().unique())
    new_levels = test_levels - train_levels
    if new_levels:
        unseen_rows.append(
            {
                "column": column,
                "n_unseen": len(new_levels),
                "examples": " | ".join(sorted(map(str, new_levels))[:5]),
            }
        )

unseen = pd.DataFrame(unseen_rows).sort_values("n_unseen", ascending=False)
print("identifiers PID/HHID are excluded: every identifier is new in test")
print(f"feature columns with at least one test-only level: {len(unseen)}")
display(unseen)

## 9. Decision summary

The table below turns the statistics above into explicit choices for the modelling
pipeline. It is a reading aid, not an automated preprocessing step.

| Observation | Consequence |
|---|---|
| `PROXY` and `COUNTRY` are constant | Drop them; they carry no information. |
| `AC1A` is 99.0% one value; `IN13` is 99.9% one value | Keep only if domain knowledge justifies it; otherwise drop or treat as a rare category. |
| Only `IN16` contains real `NaN` (51.1%) | Do not impute blindly; preserve absence and decide whether the field is usable. |
| `"Sem resposta"` / `"Não aplicável"` are widespread | Keep them as explicit categories; they encode non-response and questionnaire routing. |
| `PID`/`HHID` are unique per row and do not overlap with test | Treat as identifiers, never as features; `HHID` does not group households in this export. |
| `PE9`, `IN57`, `IN55`, `IN56`, `BM2`, `PE7` are high-cardinality | Avoid one-hot; parse only after checking the raw code format. |
| Some numeric-looking strings have leading zeros (`PE7`, `PE9`) | Do not cast blindly; preserve the fixed-width code or parse with a documented rule. |
| `HHNBPERS_COD`, `HHNBPERS_0_13_COD` mix numbers and `">=7"`/`">=3"` | Parse as ordinal with an explicit bucket rule, not as raw numeric. |
| Cramér's V is highest for `PC1E`, `PC1C`, `PC1B`, `PC1D`, `IN15`, `PC1A`, `HA1*` | Prioritise autonomy/daily-task variables for interpretation and feature engineering. |
| `MAINSTAT`, `FT_PT`, `MARSTALEGAL`, `JOBSTAT_COD` are also strongly associated | Expect socioeconomic status to carry substantial signal. |
| Test contains unseen levels in several columns | Encode with an explicit unknown bucket or a model that tolerates unseen categories. |
| Train/test TVD is small (maximum about 0.035) | No evidence of a major distribution shift; ordinary stratified validation is reasonable. |
| Classes range from 172 to 1,062 rows | Use stratification, class weights or macro-F1-aware validation; accuracy is misleading. |